# 02 — Feature Engineering

**Objetivo:** transformar el dataset OHLCV crudo (output del notebook 01) en un *Feature Store* listo para modelado, computando siete features financieras vectorizadas, definiendo la variable objetivo, aplicando normalización cross-sectional y persistiendo en Parquet con metadata de reproducibilidad.

---

### Pipeline del notebook

1. Configuración del entorno (hiperparámetros + paths).
2. Carga del dataset OHLCV desde Parquet.
3. Tratamiento de huecos con `ffill` limitado.
4. Cálculo de features (Z-Score, Amihud, Garman-Klass, RVOL, Momentum, Vol. Idiosincrásica, VIX).
5. Definición del target (retorno forward 1 día).
6. Consolidación a formato panel `(Date, Ticker)`.
7. Normalización cross-sectional (rank percentil por fecha).
8. Persistencia en Parquet + metadata JSON.
9. Verificación y estadísticas descriptivas.

### Entradas y salidas

| Tipo | Archivo |
|------|---------|
| Input  | `data/massive_financial_data.parquet` |
| Output | `data/feature_store.parquet` |
| Output | `data/feature_store_metadata.json` |

### Cambios respecto al prototipo inicial

- Migración de `os.path` → `pathlib`.
- Migración de carga CSV → Parquet (consistente con notebook 01).
- Corrección matemática de la fórmula de Garman-Klass.
- Sustitución de epsilons dispersos (`1e-10`, `1e-15`) por una constante `EPS` centralizada.
- Sustitución de `ffill()` global por `ffill(limit=2)` para evitar la generación de retornos artificiales.
- Definición explícita del **target** (ausente en el prototipo).
- Incorporación del **VIX** como feature de régimen macro (descargado en notebook 01 pero nunca usado).
- Adición de **rank cross-sectional** por fecha (decisión metodológica clave para *quant equity*).
- Logging estructurado y metadata JSON.


## 1. Configuración del entorno

Cargamos las librerías, definimos hiperparámetros financieros y rutas.

**Hiperparámetros clave:**

- `ROLLING_WINDOW = 20` — Ventana de 20 días hábiles (≈ 1 mes calendario). Estándar de la industria para Z-Score, momentum, RVOL y volatilidades rodantes.
- `TRADING_DAYS_YEAR = 252` — Para anualizar volatilidades.
- `EPS = 1e-12` — Centraliza la estabilidad numérica en divisiones (sustituye los `1e-10` y `1e-15` dispersos del prototipo).
- `FFILL_LIMIT = 2` — Máximo de días consecutivos a rellenar (ver sección 3).

**Resolución de rutas robusta:** la función `_find_project_root` busca el directorio `data/` partiendo del directorio de trabajo y subiendo hasta 3 niveles. Así el notebook funciona correctamente independientemente de si se ejecuta desde la raíz del proyecto o desde una subcarpeta (`notebooks/`, `scripts/`, etc.), evitando los problemas de hardcodear `Path.cwd().parent`.


In [1]:
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Logging estructurado consistente con el notebook 01
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("feature_engineering")

In [2]:
# --- Hiperparámetros financieros ---
ROLLING_WINDOW    = 20      # 20 días hábiles ≈ 1 mes calendario
TRADING_DAYS_YEAR = 252     # estándar para anualizar volatilidades
EPS               = 1e-12   # estabilidad numérica en divisiones
FFILL_LIMIT       = 2       # máximo de días consecutivos rellenables

# --- Universo y benchmark ---
BENCHMARK  = "^GSPC"
VIX_TICKER = "^VIX"

# --- Resolución robusta de la raíz del proyecto ---
# Busca el directorio 'data/' en cwd o subiendo hasta 2 niveles. Así funciona
# tanto si el notebook se ejecuta desde la raíz del proyecto como desde una
# subcarpeta (notebooks/, scripts/, etc.) sin necesidad de hardcodear paths.
def _find_project_root(marker: str = "data", max_levels: int = 3) -> Path:
    here = Path.cwd()
    for candidate in [here, *list(here.parents)[:max_levels]]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(
        f"No se encontró el directorio '{marker}/' partiendo de {here}. "
        "Asegúrate de ejecutar el notebook desde el proyecto bachelor-thesis."
    )

PROJECT_ROOT = _find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"

INPUT_FILE_PATH    = DATA_DIR / "massive_financial_data.parquet"
OUTPUT_FILE_PATH   = DATA_DIR / "feature_store.parquet"
METADATA_FILE_PATH = DATA_DIR / "feature_store_metadata.json"

logger.info(f"PROJECT_ROOT : {PROJECT_ROOT}")
logger.info(f"INPUT_FILE   : {INPUT_FILE_PATH}")
logger.info(f"OUTPUT_FILE  : {OUTPUT_FILE_PATH}")

# Verificación temprana: el archivo de entrada debe existir
if not INPUT_FILE_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {INPUT_FILE_PATH}. "
        "Ejecuta primero el notebook 01 para generar el dataset."
    )

12:07:26 | INFO     | PROJECT_ROOT : c:\Users\Usuario\Desktop\bachelor-thesis
12:07:26 | INFO     | INPUT_FILE   : c:\Users\Usuario\Desktop\bachelor-thesis\data\massive_financial_data.parquet
12:07:26 | INFO     | OUTPUT_FILE  : c:\Users\Usuario\Desktop\bachelor-thesis\data\feature_store.parquet


## 2. Carga del dataset OHLCV

Tras la migración del notebook 01, los datos se almacenan en **Parquet**, lo que aporta:

- Lectura ~5–10× más rápida que el CSV original.
- Preservación nativa del MultiIndex `(Ticker × Métrica)` sin ambigüedad de tipos. Esto elimina el `DtypeWarning` que aparecía al cargar el CSV con `header=[0, 1]`.
- Tipos numéricos limpios (`float64`) garantizados por el `astype` aplicado en la persistencia.

Tras cargar, separamos las cinco series OHLCV con `.xs()` y verificamos la presencia del benchmark (`^GSPC`) y del VIX (`^VIX`).


In [3]:
# Carga del Parquet generado por el notebook 01.
# Parquet preserva el MultiIndex sin ambigüedades de tipos como sí ocurría con CSV.
raw_data = pd.read_parquet(INPUT_FILE_PATH)

logger.info(f"Datos cargados: {raw_data.shape}")
logger.info(
    f"Rango temporal: {raw_data.index.min().date()} → {raw_data.index.max().date()}"
)

# Extracción vectorizada de cada componente OHLCV.
# .xs() con level=1 selecciona la métrica conservando los tickers como columnas.
open_p = raw_data.xs("Open",   axis=1, level=1)
high_p = raw_data.xs("High",   axis=1, level=1)
low_p  = raw_data.xs("Low",    axis=1, level=1)
close  = raw_data.xs("Close",  axis=1, level=1)
volume = raw_data.xs("Volume", axis=1, level=1)

logger.info(f"Activos detectados (incl. benchmark e índices): {close.shape[1]}")

# Verificación: presencia del benchmark (necesario para la vol. idiosincrásica)
assert BENCHMARK in close.columns, f"Falta el benchmark {BENCHMARK} en el dataset."
logger.info(f"Benchmark {BENCHMARK} presente.")

# VIX: si está, lo usaremos como feature macro en la sección 4.7
has_vix = VIX_TICKER in close.columns
logger.info(f"VIX {VIX_TICKER} presente: {has_vix}")

12:07:27 | INFO     | Datos cargados: (5295, 2525)
12:07:27 | INFO     | Rango temporal: 2005-01-03 → 2025-12-30
12:07:27 | INFO     | Activos detectados (incl. benchmark e índices): 505
12:07:27 | INFO     | Benchmark ^GSPC presente.
12:07:27 | INFO     | VIX ^VIX presente: True


## 3. Tratamiento de huecos: forward fill con límite

El `ffill()` global del prototipo inicial (`raw_data.ffill()`) tiene un problema sutil pero relevante en datos financieros: si un ticker **deja de cotizar** durante un periodo largo (delisting, halt prolongado, ausencia de datos en Yahoo), el forward fill arrastra el último precio observado durante semanas o meses, generando:

1. **Retornos artificiales de 0** en `pct_change()` durante todo el hueco.
2. **Volatilidades artificialmente bajas** (Z-Score, Garman-Klass, idiosincrásica).
3. Una **señal sintética** que el modelo trataría como real, contaminando el aprendizaje.

**Solución adoptada:** `ffill(limit=2)`. Permite cubrir huecos breves de 1–2 días (festivos parciales o errores leves de Yahoo Finance) pero deja como `NaN` los huecos largos, que serán filtrados por `dropna()` en la sección 6.

Es una decisión metodológica conservadora pero defensible: preferimos perder filas a inventar señales.


In [4]:
# Forward fill limitado sobre cada componente OHLCV
before_nans = close.isna().sum().sum()

open_p = open_p.ffill(limit=FFILL_LIMIT)
high_p = high_p.ffill(limit=FFILL_LIMIT)
low_p  = low_p.ffill(limit=FFILL_LIMIT)
close  = close.ffill(limit=FFILL_LIMIT)
volume = volume.ffill(limit=FFILL_LIMIT)

after_nans = close.isna().sum().sum()

logger.info(f"Forward fill aplicado (limit={FFILL_LIMIT}).")
logger.info(f"NaN en Close: {before_nans:,} → {after_nans:,}")

12:07:27 | INFO     | Forward fill aplicado (limit=2).
12:07:27 | INFO     | NaN en Close: 247,614 → 241,081


## 4. Cálculo de features

Se computan **siete features** vectorizadas. Todas las operaciones son matriciales (NumPy/pandas) — sin bucles Python — apoyándose en BLAS a nivel C para máxima eficiencia.

| Feature | Tipo de señal | Ventana | Rationale |
|---|---|---|---|
| `Z_Score` | Reversión a la media | 20 | Desviación del precio respecto a su media reciente. |
| `Amihud_Ratio` | Iliquidez | 20 | Coste de impacto: \|return\| / dollar volume. |
| `Garman_Klass_Vol` | Volatilidad intradía | 20 | Estimador eficiente que usa OHLC (no solo Close). |
| `RVOL` | Volumen relativo | 20 | Detecta interés institucional anómalo. |
| `Momentum_1M` | Tendencia | 20 | Persistencia de retornos a 1 mes. |
| `Idiosyncratic_Vol` | Riesgo específico | 20 | Volatilidad neutralizada al mercado (CAPM rodante). |
| `VIX` / `VIX_ZScore` | Régimen macro | — | Volatilidad implícita del S&P 500 (broadcast a todo el panel). |


### 4.1 Z-Score (reversión a la media)

Mide cuántas desviaciones estándar se aleja el precio actual respecto a su media móvil de 20 días:

$$
Z_t = \frac{P_t - \mu_{t,20}}{\sigma_{t,20}}
$$

Valores $|Z| > 2$ se interpretan tradicionalmente como zonas de sobreventa/sobrecompra. Para evitar divisiones por cero en activos con volatilidad nula puntual, añadimos `EPS` al denominador.


In [5]:
rolling_mean = close.rolling(window=ROLLING_WINDOW).mean()
rolling_std  = close.rolling(window=ROLLING_WINDOW).std()
z_score = (close - rolling_mean) / (rolling_std + EPS)

### 4.2 Ratio de Amihud (iliquidez)

Estimador de iliquidez propuesto por Yakov Amihud (2002): cuánto se mueve el precio relativo al volumen negociado en dólares.

$$
ILLIQ_t = \frac{|R_t|}{V_t \cdot P_t}
$$

donde el denominador es el *dollar volume* (volumen × precio). Promediamos sobre la ventana móvil para suavizar. Los activos con ratio alto requieren mayor coste de impacto al operar — son menos líquidos y por tanto más propensos a *slippage*.


In [6]:
returns       = close.pct_change()
returns_abs   = returns.abs()
dollar_volume = close * volume

amihud = (returns_abs / (dollar_volume + EPS)).rolling(ROLLING_WINDOW).mean()

### 4.3 Volatilidad Garman-Klass (intradía)

Estimador de Garman & Klass (1980), aproximadamente **8× más eficiente** que el estimador close-to-close estándar al usar los cuatro precios OHLC en lugar de solo el Close:

$$
\sigma^2_{GK,t} = \tfrac{1}{2}\,\ln\!\left(\frac{H_t}{L_t}\right)^{\!2} - (2\ln 2 - 1)\,\ln\!\left(\frac{C_t}{O_t}\right)^{\!2}
$$

#### Correcciones aplicadas frente al prototipo inicial

1. **Eliminación de epsilons espurios.** El prototipo añadía `+ 1e-10` *fuera* del logaritmo:
   ```python
   np.log((high_p / (low_p + 1e-10)) + 1e-10) ** 2   # ← incorrecto
   ```
   Esto es matemáticamente erróneo: distorsiona el ratio en lugar de estabilizarlo. Como los precios de activos cotizando en el S&P 500 no son nunca cero, los epsilons no son necesarios y se eliminan.

2. **Truncamiento de varianzas negativas.** El estimador puede producir varianzas negativas cuando $\ln(C/O)^2$ domina (gaps de apertura grandes — overnight news, earnings). Truncamos a cero antes de aplicar la raíz cuadrada (forma estándar de manejar esta limitación):
   ```python
   gk_variance = gk_variance.clip(lower=0)
   ```

Finalmente, anualizamos la varianza promediada multiplicando por $\sqrt{252}$.


In [7]:
# Logaritmos al cuadrado de los ratios H/L y C/O
log_hl_sq = np.log(high_p / low_p)  ** 2
log_co_sq = np.log(close  / open_p) ** 2

# Varianza Garman-Klass (puede ser negativa; ver markdown)
gk_variance = 0.5 * log_hl_sq - (2 * np.log(2) - 1) * log_co_sq
gk_variance = gk_variance.clip(lower=0)

# Promedio rodante y anualización
gk_volatility = np.sqrt(gk_variance.rolling(ROLLING_WINDOW).mean() * TRADING_DAYS_YEAR)

### 4.4 RVOL (volumen relativo)

Volumen de la sesión actual normalizado por el volumen medio de los últimos 20 días:

$$
RVOL_t = \frac{V_t}{\overline{V}_{t,20}}
$$

Valores $RVOL > 1.5$ suelen indicar interés institucional anómalo (acumulación o distribución), y son señales de movimiento direccional inminente. Es un indicador favorito de los traders de *day trading* institucional.


In [8]:
volume_ma = volume.rolling(window=ROLLING_WINDOW).mean()
rvol = volume / (volume_ma + EPS)

### 4.5 Momentum a 1 mes

Retorno acumulado a 20 días hábiles:

$$
M_t = \frac{P_t}{P_{t-20}} - 1
$$

Captura la **persistencia de tendencias** documentada en la literatura factor (Jegadeesh & Titman, 1993). Es uno de los pilares del *factor investing* moderno y aparece en el modelo de Fama-French extendido (Fama-French + Momentum).


In [9]:
momentum_1m = close.pct_change(periods=ROLLING_WINDOW)

### 4.6 Volatilidad idiosincrásica

Volatilidad **neutralizada al mercado**: la parte del riesgo del activo que no se explica por el movimiento del benchmark. Procedimiento (rolling CAPM):

1. **Beta rodante:**
$$
\beta_t = \frac{\mathrm{Cov}(R_i, R_m)_{t,20}}{\mathrm{Var}(R_m)_{t,20}}
$$
2. **Retorno esperado:**
$$
\hat{R}_{i,t} = \beta_t \cdot R_{m,t}
$$
3. **Residuo (alpha diario):**
$$
\varepsilon_t = R_{i,t} - \hat{R}_{i,t}
$$
4. **Volatilidad idiosincrásica anualizada:**
$$
\sigma_{idio,t} = \sqrt{252}\cdot\sigma(\varepsilon)_{t,20}
$$

Es una de las features más informativas en *quant equity*: un activo con volatilidad idiosincrásica alta ofrece *alpha potencial* descorrelacionado del mercado.

> **Nota:** se ha corregido el comentario del prototipo, que decía *"Calculo de la volatilidad ideosincrónica (RVOL) para el benchmark"* mezclando dos conceptos distintos (RVOL e idiosincrásica) y con error ortográfico.


In [10]:
benchmark_returns = returns[BENCHMARK]

# 1. Varianza rodante del mercado
var_market = benchmark_returns.rolling(ROLLING_WINDOW).var()

# 2. Covarianza rodante de cada activo frente al mercado.
#    Pasar una Serie (benchmark) a un DataFrame.rolling.cov() alinea automáticamente.
cov_assets_market = returns.rolling(ROLLING_WINDOW).cov(benchmark_returns)

# 3. Beta rodante: sensibilidad de cada activo al mercado
rolling_beta = cov_assets_market.div(var_market + EPS, axis=0)

# 4. Retorno esperado según CAPM rodante: R_i_hat = β · R_m
expected_returns = rolling_beta.multiply(benchmark_returns, axis=0)

# 5. Residuos: lo que no explica el mercado (alpha diario)
residuals = returns - expected_returns

# 6. Volatilidad idiosincrásica = std anualizada de los residuos
idio_vol = residuals.rolling(ROLLING_WINDOW).std() * np.sqrt(TRADING_DAYS_YEAR)

### 4.7 VIX (régimen de mercado macro)

El **VIX** (CBOE Volatility Index) mide la volatilidad implícita anualizada del S&P 500 a 30 días según el mercado de opciones. Es conocido como el *"índice del miedo"*: valores altos indican estrés de mercado.

A diferencia de las features anteriores, el VIX **no es específico de cada activo**: es una variable macro que se *broadcasta* al panel — todos los activos comparten el mismo valor de VIX en una fecha dada. Esto permite al modelo condicionar las señales según el contexto macro (p. ej. momentum funciona mejor en regímenes tranquilos; reversión a la media domina en regímenes estresados).

Se añaden **dos versiones**:

- **`VIX`**: valor crudo del índice.
- **`VIX_ZScore`**: Z-Score 20d del VIX para detectar regímenes anómalos (alto VIX en relación a su histórico reciente).

> El VIX se descargó en el notebook 01 pero quedó sin usar; aquí se incorpora al pipeline.


In [11]:
if has_vix:
    vix_series = close[VIX_TICKER]   # Serie indexada por fecha

    # Z-Score del VIX para detectar regímenes anómalos
    vix_mean   = vix_series.rolling(ROLLING_WINDOW).mean()
    vix_std    = vix_series.rolling(ROLLING_WINDOW).std()
    vix_zscore = (vix_series - vix_mean) / (vix_std + EPS)

    logger.info(f"VIX procesado. Rango: [{vix_series.min():.2f}, {vix_series.max():.2f}]")
else:
    vix_series = None
    vix_zscore = None
    logger.warning("VIX no disponible — se omiten las features macro.")

12:07:28 | INFO     | VIX procesado. Rango: [9.14, 82.69]


## 5. Definición del target

Definimos el objetivo de predicción: el **retorno simple a 1 día vista**.

$$
y_t = \frac{P_{t+1} - P_t}{P_t}
$$

En código: `close.pct_change().shift(-1)`. Equivalentemente: en la fecha $t$, las features están construidas con información hasta $\text{close}[t]$, y predecimos el retorno realizado entre $t$ y $t+1$. Esto garantiza que **no hay look-ahead bias**: nunca usamos información futura para construir las features de hoy.

> Para clasificación binaria direccional (sube/baja), basta con derivar `(target > 0).astype(int)` en el notebook 03. Mantenemos aquí el target continuo para máxima flexibilidad — preserva la información sobre la magnitud del movimiento, no solo el signo.


In [12]:
# Retorno forward a 1 día. shift(-1) lleva el retorno de t+1 a la fila t,
# permitiendo alinear correctamente target con features.
forward_returns_1d = close.pct_change().shift(-1)

logger.info(f"Target calculado. Shape: {forward_returns_1d.shape}")

12:07:28 | INFO     | Target calculado. Shape: (5295, 505)


## 6. Consolidación del Feature Store

Pasamos del formato *wide* (DataFrame `Date × Ticker` por feature) al formato *long* o **panel** (`(Date, Ticker)` como MultiIndex, features como columnas). Es el formato canónico para entrenamiento de modelos sobre datos de panel.

**Operaciones realizadas:**

1. **`stack(future_stack=True)`** sobre cada feature: pivota tickers de columnas a índice. El flag `future_stack` evita el comportamiento legacy que descartaba filas con todos los valores `NaN` (lo necesitamos preservar para no romper la alineación).
2. **Eliminación del benchmark y del VIX como activos predecibles**: el modelo no debe predecir `^GSPC` ni `^VIX`. Estos solo entran como inputs (volatilidad idiosincrásica y feature macro), no como targets.
3. **Broadcast del VIX al panel completo**: cada par `(Date, Ticker)` recibe el valor de VIX correspondiente a esa fecha.
4. **Limpieza de `NaN`**: filas con cualquier feature/target faltante son descartadas. Es una decisión conservadora pero defensible para un primer modelo.


In [13]:
# 1. Stack a formato panel (Date × Ticker como índice)
features = pd.DataFrame({
    "Z_Score":           z_score.stack(future_stack=True),
    "Amihud_Ratio":      amihud.stack(future_stack=True),
    "Garman_Klass_Vol":  gk_volatility.stack(future_stack=True),
    "RVOL":              rvol.stack(future_stack=True),
    "Momentum_1M":       momentum_1m.stack(future_stack=True),
    "Idiosyncratic_Vol": idio_vol.stack(future_stack=True),
    "Target_FwdRet_1D":  forward_returns_1d.stack(future_stack=True),
})
features.index.names = ["Date", "Ticker"]

# 2. Eliminar índices del panel (no son activos predecibles)
non_assets = [BENCHMARK] + ([VIX_TICKER] if has_vix else [])
features = features.drop(non_assets, level="Ticker", errors="ignore")
logger.info(f"Tras eliminar {non_assets}: {features.shape}")

# 3. Broadcast del VIX como feature macro
if has_vix:
    dates = features.index.get_level_values("Date")
    features["VIX"]        = vix_series.reindex(dates).values
    features["VIX_ZScore"] = vix_zscore.reindex(dates).values
    logger.info("Features macro VIX y VIX_ZScore añadidas al panel.")

# 4. Limpieza de NaNs
n_before = len(features)
features = features.dropna()
n_after  = len(features)
logger.info(f"Filas tras dropna: {n_after:,} (eliminadas {n_before - n_after:,})")
logger.info(f"Feature store base: {features.shape}")

12:07:32 | INFO     | Tras eliminar ['^GSPC', '^VIX']: (2663385, 7)
12:07:32 | INFO     | Features macro VIX y VIX_ZScore añadidas al panel.
12:07:32 | INFO     | Filas tras dropna: 2,402,184 (eliminadas 261,201)
12:07:32 | INFO     | Feature store base: (2402184, 9)


## 7. Normalización cross-sectional (rank por fecha)

**Decisión metodológica clave para la calidad del modelo.**

Los valores absolutos de muchas features cambian con el régimen de mercado: la distribución del `Z_Score` en 2008 (crisis financiera) y en 2017 (mercado tranquilo) son estadísticamente distintas. Si entrenamos el modelo con valores crudos, aprenderá *patrones de régimen* en lugar de la **señal relativa entre activos** del mismo día.

**Solución:** convertir cada feature al **rank percentil entre todos los activos de la misma fecha** (`groupby(Date).rank(pct=True)`). Tras la transformación, cada feature está acotada en $[0, 1]$ y representa la posición relativa del activo dentro del universo en ese día.

**Beneficios:**

- Elimina el efecto régimen (no-stationarity).
- Hace al modelo invariante al cambio de unidades.
- Es la práctica estándar en *quant equity* (literatura clásica: Lo & MacKinlay; portfolios largo-corto basados en *deciles* de la señal).

Conservamos en el feature store **ambas versiones** (raw + ranked, esta última con sufijo `_rank`) para que el notebook 03 pueda elegir o combinar. El target permanece sin transformar.


In [14]:
# Identificar columnas a rankear: todas las features, NO el target
feature_cols = [c for c in features.columns if not c.startswith("Target")]

# Rank percentil [0, 1] por fecha. pct=True devuelve directamente el percentil.
ranked = (
    features[feature_cols]
    .groupby(level="Date")
    .rank(pct=True)
    .add_suffix("_rank")
)

# Concatenar versión raw y versión rankeada en el mismo dataframe
features_full = pd.concat([features, ranked], axis=1)

logger.info(f"Feature store final (raw + ranked + target): {features_full.shape}")
logger.info(f"Columnas: {features_full.columns.tolist()}")

12:07:39 | INFO     | Feature store final (raw + ranked + target): (2402184, 17)
12:07:39 | INFO     | Columnas: ['Z_Score', 'Amihud_Ratio', 'Garman_Klass_Vol', 'RVOL', 'Momentum_1M', 'Idiosyncratic_Vol', 'Target_FwdRet_1D', 'VIX', 'VIX_ZScore', 'Z_Score_rank', 'Amihud_Ratio_rank', 'Garman_Klass_Vol_rank', 'RVOL_rank', 'Momentum_1M_rank', 'Idiosyncratic_Vol_rank', 'VIX_rank', 'VIX_ZScore_rank']


## 8. Persistencia + metadata

Guardamos el feature store en **Parquet** (formato eficiente, preserva el MultiIndex) y un fichero JSON con metadata para reproducibilidad: timestamp de generación, dimensiones, hiperparámetros usados, versiones de librerías y listas de features y targets.

Este patrón replica el establecido en el notebook 01 — toda fase del pipeline genera su correspondiente fichero `*_metadata.json`.


In [15]:
def save_feature_metadata(path: Path, features_df: pd.DataFrame) -> None:
    """Persiste metadata del feature store para reproducibilidad."""
    feature_cols = [c for c in features_df.columns if not c.startswith("Target")]
    target_cols  = [c for c in features_df.columns if c.startswith("Target")]

    metadata = {
        "build_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "rows":               int(features_df.shape[0]),
        "columns":            int(features_df.shape[1]),
        "n_unique_dates":     int(features_df.index.get_level_values("Date").nunique()),
        "n_unique_tickers":   int(features_df.index.get_level_values("Ticker").nunique()),
        "date_range": {
            "start": str(features_df.index.get_level_values("Date").min().date()),
            "end":   str(features_df.index.get_level_values("Date").max().date()),
        },
        "features": feature_cols,
        "targets":  target_cols,
        "hyperparams": {
            "rolling_window":    ROLLING_WINDOW,
            "trading_days_year": TRADING_DAYS_YEAR,
            "ffill_limit":       FFILL_LIMIT,
            "epsilon":           EPS,
        },
        "library_versions": {
            "pandas": pd.__version__,
            "numpy":  np.__version__,
        },
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    logger.info(f"Metadata guardada en: {path}")


# Persistencia
OUTPUT_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
features_full.to_parquet(OUTPUT_FILE_PATH, engine="pyarrow", compression="snappy")
logger.info(f"Feature store guardado en: {OUTPUT_FILE_PATH}")

save_feature_metadata(METADATA_FILE_PATH, features_full)

12:07:40 | INFO     | Feature store guardado en: c:\Users\Usuario\Desktop\bachelor-thesis\data\feature_store.parquet
12:07:40 | INFO     | Metadata guardada en: c:\Users\Usuario\Desktop\bachelor-thesis\data\feature_store_metadata.json


## 9. Verificación

Sanity check final: dimensiones, rango temporal, estadísticas descriptivas y cabecera del DataFrame antes de pasar al notebook `03-model-training.ipynb`.

Cosas a comprobar visualmente:

- Las features rankeadas (`*_rank`) deben estar en $[0, 1]$.
- `Target_FwdRet_1D` debe tener media próxima a cero y rango razonable (típicamente $\pm 0.1$ para retornos diarios).
- No debe haber `NaN` (todos eliminados en la sección 6).


In [16]:
dates_idx   = features_full.index.get_level_values("Date")
tickers_idx = features_full.index.get_level_values("Ticker")

print(f"Forma del Feature Store     : {features_full.shape}")
print(f"Tickers únicos              : {tickers_idx.nunique()}")
print(f"Fechas únicas               : {dates_idx.nunique()}")
print(f"Rango temporal              : {dates_idx.min().date()} → {dates_idx.max().date()}")
print(f"Memoria en RAM (MB)         : {features_full.memory_usage(deep=True).sum() / 1024**2:.2f}")
print(f"NaNs totales                : {features_full.isna().sum().sum()}")

print("\n--- Estadísticas descriptivas ---")
print(features_full.describe().round(4).T)

print("\n--- Cabecera ---")
print(features_full.head())

Forma del Feature Store     : (2402184, 17)
Tickers únicos              : 503
Fechas únicas               : 5255
Rango temporal              : 2005-03-01 → 2025-12-29
Memoria en RAM (MB)         : 320.81
NaNs totales                : 0

--- Estadísticas descriptivas ---
                            count         mean           std     min      25%  \
Z_Score                 2402184.0       0.1997  1.291900e+00 -4.2485  -0.8345   
Amihud_Ratio            2402184.0  225660.2759  2.288275e+07  0.0000   0.0000   
Garman_Klass_Vol        2402184.0       0.2585  1.543000e-01  0.0000   0.1702   
RVOL                    2402184.0       1.0114  5.200000e-01  0.0000   0.7265   
Momentum_1M             2402184.0       0.0132  9.220000e-02 -0.8981  -0.0328   
Idiosyncratic_Vol       2402184.0       0.2249  1.488000e-01  0.0000   0.1361   
Target_FwdRet_1D        2402184.0       0.0007  2.200000e-02 -0.6079  -0.0085   
VIX                     2402184.0      19.1491  8.531000e+00  9.1400  13.5300   


---

### Nota técnica para la memoria del TFG

El feature engineering implementado opera enteramente en operaciones vectorizadas de NumPy/pandas, que delegan en BLAS/LAPACK a nivel C. Esto reduce la complejidad efectiva desde un naive $O(N \times M \times T)$ con triple bucle Python — donde $N$ es el número de tickers (~503), $M$ las features (7) y $T$ los días (~5275) — a operaciones matriciales paralelizables en SIMD.

En la práctica: procesamos **~2.4 millones de observaciones × 9 columnas** (raw + ranked + target) en pocos segundos, frente a las horas que requeriría una implementación iterativa equivalente. Es uno de los puntos fuertes del proyecto desde la perspectiva de Ingeniería Informática y conviene destacarlo en la memoria.
